In [1]:
"""
Created on July 15 2025
@author: AF

Function: process_tributary_flow_data
-------------------------------------
Automates hydrologic data processing for tributary flow analysis.

Main steps:
- Download USGS discharge and stage data
- Read in Corps discharge data
- Fill missing data:
    - Short-duration missing values: linear interpolation 
    - Long-duration missing values: rating curve or linear interpolation if rating curve does not exist
- Save discharge data in CSV file
- Plot discharge data
"""

# ---------------------------------------------------------------
# Imports and Global Constants
# ---------------------------------------------------------------

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Utility functions for fetching/reindexing/gap-filling
import flow_data_utils as fdu

# Paths for metadata and data inputs 
METADATA_EXCEL_PATH = os.path.join("Tributary_Flow_Stations_and_Rating_Curves.xlsx")
DATA_RAW_DIR = "data_raw"
USACE_DIR = os.path.join(DATA_RAW_DIR, "USACE")
CPRA_DIR  = os.path.join(DATA_RAW_DIR, "CPRA")

# Processing parameters
SHORT_DURATION_DAYS = 3  # Max consecutive missing days for linear interpolation
START_DATE = "2006-01-01"
END_DATE = "2025-09-01"
PARAMETER_CODES = ["00060", "00065"]  # 00060=Discharge, 00065=Gage height

# Output/plot settings
PLOT_OUTDIR = "plots"
PLOT_DPI = 300
PLOT_FORMAT = "png"
DATA_OUTDIR = "data_csv"
CACHE_OUTDIR = "cache"
CACHE_FILE = os.path.join(CACHE_OUTDIR, "tributary_cache.joblib")
os.makedirs(CACHE_OUTDIR, exist_ok=True)

# Ensure output directories exist
os.makedirs(PLOT_OUTDIR, exist_ok=True)
os.makedirs(DATA_OUTDIR, exist_ok=True)

In [3]:
# ---------------------------------------------------------------
# Cell 2 — Load Station Metadata
# ---------------------------------------------------------------

# Read Excel as strings to preserve IDs
dtype_dict = {"Station ID": str, "Station ID used for rating curve": str}
df_metadata = pd.read_excel(METADATA_EXCEL_PATH, dtype=dtype_dict)

# Clean the ID columns
for col in ["Station ID", "Station ID used for rating curve"]:
    if col in df_metadata.columns:
        df_metadata[col] = df_metadata[col].apply(lambda x: str(x).strip() if pd.notna(x) else "")

station_metadata = {}
station_names = {}

for _, row in df_metadata.iterrows():
    station_id = row.get("Station ID", "")
    if not station_id:
        continue

    station_metadata[station_id] = row.to_dict()
    station_names[station_id] = row.get("Station", "")

print(f"Loaded stations: {len(station_metadata)}")

Loaded stations: 34


In [5]:
# ---------------------------------------------------------------
# Download USGS data and organize by station_id
# ---------------------------------------------------------------

flow_data = {}  # raw daily time series dataframe

for station_id, metadata in station_metadata.items():
    # Only process USGS stations in this loop
    if str(metadata.get("Agency", "")).upper() != "USGS":
        continue

    # Download daily discharge (00060) and stage (00065) data for time period
    df = fdu.fetch_usgs_data_dataretrieval(
        station_id,
        start_date=START_DATE,
        end_date=END_DATE,
        parameter_codes=PARAMETER_CODES
    )

    # Skip stations with no data
    if df is None or df.empty or "Date" not in df.columns:
        continue

    # Store raw data
    flow_data[station_id] = df

# Summary of downloaded USGS stations
print(f"Fetched USGS stations: {len(flow_data)}")

Reading in station 07381490 ...
  -> Fetching instantaneous data for 2328 missing days...
Reading in station 295501090190400 ...
  -> Fetching instantaneous data for 7184 missing days...
Reading in station 295124089542100 ...
  -> Fetching instantaneous data for 3930 missing days...
Reading in station 02470629 ...
  -> Fetching instantaneous data for 7184 missing days...
Reading in station 02471019 ...
  -> Fetching instantaneous data for 7184 missing days...
Reading in station 02479000 ...
  -> Fetching instantaneous data for 93 missing days...
Reading in station 02481000 ...
  -> Fetching instantaneous data for 148 missing days...
Reading in station 02481510 ...
  -> Fetching instantaneous data for 69 missing days...
Reading in station 02489500 ...
  -> Fetching instantaneous data for 49 missing days...
Reading in station 02492000 ...
  -> Fetching instantaneous data for 26 missing days...
Reading in station 07375000 ...
  -> Fetching instantaneous data for 718 missing days...
Readin

In [8]:
# ---------------------------------------------------------------
# Load USACE Tarbert Landing discharge CSV and convert to daily m3/s
# ---------------------------------------------------------------

def load_usace_tarbert_csv(usace_dir=USACE_DIR, pattern="*01100*.csv"):
    import glob

    # Find matching Tarbert CSV
    paths = glob.glob(os.path.join(usace_dir, pattern))
    if not paths:
        print(f"USACE CSV not found in {usace_dir} with pattern {pattern}")
        return None

    # Skip USACE header block
    df = pd.read_csv(paths[0], skiprows=16, encoding="cp1252", thousands=",")

    # Find date column and discharge columns
    date_col = next((c for c in df.columns if "date" in str(c).lower() or "time" in str(c).lower()), None)
    flow_col = next((c for c in df.columns if "flow" in str(c).lower() or "discharge" in str(c).lower()), None)

    # Error if CSV structure not found
    if not date_col or not flow_col:
        print("USACE: Date/Flow columns not found")
        return None

    # Keep only required columns
    df = df[[date_col, flow_col]].copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(by=date_col)

    # Convert CFS to m3/s to match USGS units
    q = pd.to_numeric(df[flow_col], errors="coerce")
    df["Discharge"] = q * 0.0283168

    # Convert to daily dates
    df["Date"] = df[date_col].dt.normalize()

    # Daily mean
    return df.groupby("Date", as_index=False)["Discharge"].mean()[["Date", "Discharge"]]

In [10]:
# ---------------------------------------------------------------
# Integrate USACE Tarbert Landing (01100) into flow_data
# ---------------------------------------------------------------

# load daily USACE discharge with expected columns Date and Discharge)
usace_01100 = load_usace_tarbert_csv()

if usace_01100 is not None:
    flow_data["01100"] = usace_01100

    # Ensure station name exists
    station_names.setdefault("01100", station_metadata["01100"]["Station"])

    #Print status
    print(f"Integrated USACE 01100 with {len(usace_01100)} daily rows")

Integrated USACE 01100 with 7174 daily rows


In [11]:
# ---------------------------------------------------------------
# Reindex all station time series to full date range and normalize columns
# ---------------------------------------------------------------

# Reindexed daily DataFrame
flow_data_reindexed = {}
required_cols = ["Discharge", "Stage"]

# Add missing dates from START_DATE to END_DATE
for station_id, df in flow_data.items():
    
    # Reindex to full date range
    df_reindexed = fdu.reindex_to_full_range(df, START_DATE, END_DATE)
    
    # Ensure required columns exist
    for col in required_cols:
        if col not in df_reindexed.columns:
            df_reindexed[col] = np.nan
    
    #Store reindexed data
    flow_data_reindexed[station_id] = df_reindexed

In [13]:
# ---------------------------------------------------------------
# Label existing discharge values as "original" (for CSV export)
# ---------------------------------------------------------------

labeled_total = 0

for station_id, df in flow_data_reindexed.items():
    # Skip stations with no data or without a Discharge column
    if df.empty or "Discharge" not in df.columns:
        continue

    # Work on a copy to avoid mutating a view / unexpected side effects
    df = df.copy()

    # Ensure the method/flag column exists
    if "Discharge_filled_method" not in df.columns:
        df["Discharge_filled_method"] = ""

    # Label all existing discharge values
    mask = df["Discharge"].notna()
    df.loc[mask, "Discharge_filled_method"] = "original"

    # Store updated station data back into the dict and update summary count
    flow_data_reindexed[station_id] = df
    labeled_total += int(mask.sum())

# Summary across all stations
print(f"Total labeled 'original' rows across stations: {labeled_total}")

Total labeled 'original' rows across stations: 200216


In [16]:
# ---------------------------------------------------------------
# Run the gap-filling pipeline for each station and store results.
# ---------------------------------------------------------------

filled_flow_data = {}

# Station data needed by the pipeline
for station_id, df in flow_data_reindexed.items():
    meta = station_metadata[station_id]
    station_name = station_names[station_id]

    #Run gap-filling pipeline
    filled_df = fdu.gap_filling_pipeline_with_metadata(
    df, meta, station_name, flow_data_reindexed,
    discharge_col="Discharge", stage_col="Stage"
)
    # Store filled result for export and plots
    filled_flow_data[station_id] = filled_df

In [19]:
# ---------------------------------------------------------------
# Save station DataFrame to CSV
# ---------------------------------------------------------------

def save_to_csv(df, filepath):
    # Processes on copy of DataFrame
    df_out = df.copy()

    # Remove extraneous index columns, if present
    for extra_col in ["level_0", "index"]:
        if extra_col in df_out.columns:
            df_out = df_out.drop(columns=[extra_col])

    # Rename for output only
    if "Discharge" in df_out.columns:
        df_out = df_out.rename(columns={"Discharge": "Discharge (m3/s)"})

    # Keep only specific comments, only if they exist
    cols_to_save = ["Date", "Discharge (m3/s)", "Discharge_filled_method"]
    df_out = df_out.loc[:, [c for c in cols_to_save if c in df_out.columns]]

    # Print to confirm columns being saved
    print("Saving columns:", df_out.columns.tolist(), "to", filepath)
    df_out.to_csv(filepath, index=False, encoding="utf-8")


In [20]:
# ---------------------------------------------------------------
# Export all station CSVs with filled output, and add units to 
# header 
# ---------------------------------------------------------------
for station_id, df in filled_flow_data.items():
    station_name = station_names[station_id]
   
    # Clean up the station name for filename
    clean_name = station_name.replace(" ", "_").replace(",", "").replace("/", "_")

    # Set filename
    filename = f"{clean_name}_{station_id}_filled.csv"
    filepath = os.path.join(DATA_OUTDIR, filename)

    # Save CSV file
    save_to_csv(df, filepath)

Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to data_csv\Atchafalaya_River_at_Simmesport_LA_07381490_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to data_csv\Davis_Pond_295501090190400_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to data_csv\Caernarvon_295124089542100_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to data_csv\Mobile_River_at_River_Mile_31_at_Bucks_AL_02470629_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to data_csv\Tensaw_River_near_Mount_Vernon_AL_02471019_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to data_csv\Pascagoula_River_at_Merrill_MS_02479000_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_method'] to data_csv\Biloxi_River_at_Wortham_MS_02481000_filled.csv
Saving columns: ['Date', 'Discharge (m3/s)', 'Discharge_filled_meth

In [22]:
# ---------------------------------------------------------------
# Plot Discharge and Gap-Filling Methods
# ---------------------------------------------------------------

def plot_filling_methods(
    df,
    station_name,
    station_id,
    discharge_col="Discharge",
    method_col="Discharge_filled_method",
    date_col="Date",
    figsize=(16, 6),
    outdir=PLOT_OUTDIR,
    dpi=PLOT_DPI,
    fmt=PLOT_FORMAT,
):
  
    #Plot discharge values colored by how they were filled (original/interpolated/etc.)
   
    # Work on a copy 
    df = df.copy()

    # Ensure the date column exists and is datetime
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    else:
        if df.index.name == date_col or date_col in getattr(df.index, "names", []):
            df = df.reset_index()
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        else:
            raise KeyError(f"'{date_col}' not found in columns or index of DataFrame")

    # Drop rows with invalid or missing dates to avoid plotting errors
    df = df.dropna(subset=[date_col])

    # Create a station name for the output filename
    safe_station = (
        str(station_name)
        .replace(" ", "_")
        .replace(",", "")
        .replace("/", "_")
    )

    # Output path for the figure
    fname = f"{safe_station}_{station_id}_discharge.{fmt}"
    save_path = os.path.join(outdir, fname)

    # Define color for each filling method
    method_info = {
        "original": {"color": "black", "label": "Original"},
        "interpolated": {"color": "deepskyblue", "label": "Interpolated"},
        "rating_curve_long_gap": {"color": "orange", "label": "Rating Curve"},
        "interpolated_long_gap": {"color": "purple", "label": "Long Gap Interp"},
        "unfillable_long_missing": {"color": "red", "label": "Unfillable"},
    }

    # Create a new figure and axis
    fig, ax = plt.subplots(figsize=figsize)

    # Track legend handles and labels for only the methods that appear in the data
    handles = []
    labels = []

    # Plot each method category as a separate layer
    for method, info in method_info.items():
        mask = df[method_col].astype(str) == method

        if not mask.any():
            continue

        # Plot unfillable points as red Xs at y=0 to make them visible
        if method == "unfillable_long_missing":
            sc = ax.scatter(
                df.loc[mask, date_col],
                np.zeros(int(mask.sum())),
                color=info["color"],
                label=info["label"],
                s=40,
                alpha=0.9,
                marker="x",
                zorder=5,
            )
        else:
            sc = ax.scatter(
                df.loc[mask, date_col],
                df.loc[mask, discharge_col],
                color=info["color"],
                label=info["label"],
                s=20,
                alpha=0.9,
            )

        handles.append(sc)
        labels.append(info["label"])

    # Add axis labels and a title
    ax.set_xlabel("Date")
    ax.set_ylabel("Discharge (m3/s)")
    ax.set_title(f"{station_name} ({station_id})")

    # Add legend only if at least one method category was plotted
    if handles:
        ax.legend(handles, labels, loc="upper right", markerscale=2, fontsize=14)

    # Improve layout so labels are not cut off
    fig.tight_layout()

    # Save and close the figure 
    fig.savefig(save_path, dpi=dpi)
    plt.close(fig)

    # Print saved path 
    print(f"Plot saved to {save_path}")
   

In [23]:
# ---------------------------------------------------------------
# Generate Plots for All Stations
# ---------------------------------------------------------------
for station_id, df in filled_flow_data.items():
    station_name = station_names.get(station_id, station_id)
    plot_filling_methods(df, station_name, station_id)

Plot saved to plots\Atchafalaya_River_at_Simmesport_LA_07381490_discharge.png
Plot saved to plots\Davis_Pond_295501090190400_discharge.png
Plot saved to plots\Caernarvon_295124089542100_discharge.png
Plot saved to plots\Mobile_River_at_River_Mile_31_at_Bucks_AL_02470629_discharge.png
Plot saved to plots\Tensaw_River_near_Mount_Vernon_AL_02471019_discharge.png
Plot saved to plots\Pascagoula_River_at_Merrill_MS_02479000_discharge.png
Plot saved to plots\Biloxi_River_at_Wortham_MS_02481000_discharge.png
Plot saved to plots\Wolf_River_near_Landon_MS_02481510_discharge.png
Plot saved to plots\Pearl_River_near_Bogalusa_LA_02489500_discharge.png
Plot saved to plots\Bogue_Chitto_near_Bush_LA_02492000_discharge.png
Plot saved to plots\Tchefuncte_River_near_Folsom_LA_07375000_discharge.png
Plot saved to plots\Tangipahoa_River_at_Robert_LA_07375500_discharge.png
Plot saved to plots\Tickfaw_River_at_Holden_LA_07376000_discharge.png
Plot saved to plots\Natalbany_River_at_Baptist_LA_07376500_dischar

In [24]:
# ---------------------------------------------------------------
# Cache/save processed outputs
# ---------------------------------------------------------------

# Save key outputs for quick reload/debugging
joblib.dump(
    {
        "flow_data": flow_data,  # raw downloads (USGS + USACE integrated)
        "flow_data_reindexed": flow_data_reindexed,
        "filled_flow_data": filled_flow_data,
        "station_metadata": station_metadata,
        "station_names": station_names,
    },
    CACHE_FILE,
    compress=3,
)
print("Saved cache:", CACHE_FILE)

Saved cache: cache\tributary_cache.joblib
